# 🥑 03. Modelos Predictivos de Rendimiento Agrícola y Clasificación de Calidad Exportable
**Proyecto**: AgroStats AndTech — Plataforma de Inteligencia Agrícola Colombiana  
**Capa del Lakehouse**: **Gold** (Feature Store `features_yield_prediction.parquet`)  
**Técnicas**: Random Forest Regressor, Gradient Boosting (GBR), Ridge Regression, Random Forest Classifier  
**Estándares**: ISO/IEC 25010, K-Fold Cross Validation, Permutation Feature Importance, Model Registry  

---

### Objetivos del Cuaderno:
1. **Modelado de Rendimiento (kg/ha)**: Predecir el rendimiento agrícola a partir de variables edafoclimáticas y biométricas.
2. **Benchmark de Algoritmos**: Comparar Ridge, Random Forest y Gradient Boosting mediante 5-Fold Cross Validation ($R^2$, RMSE).
3. **Interpretabilidad**: Analizar la importancia de características (MDI y Permutación) para identificar las palancas críticas de productividad (grados Brix, calibre, pH del suelo).
4. **Clasificación de Calidad Exportable**: Entrenar un clasificador supervisado para categorizar lotes en `PREMIUM_EXPORT`, `ESTANDAR_EXPORT` o `MERCADO_NACIONAL`.
5. **Serialización**: Registrar y persistir el modelo óptimo en `data/models/`.


In [1]:
import matplotlib
matplotlib.use('Agg')
# 1. Configuración de Entorno e Importación de Librerías
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

WORKSPACE_DIR = Path.cwd()
if WORKSPACE_DIR.name in ["notebooks", "03_gold_yield_ml"]:
    BASE_DIR = WORKSPACE_DIR.parents[1] if WORKSPACE_DIR.name == "03_gold_yield_ml" else WORKSPACE_DIR.parent
else:
    BASE_DIR = WORKSPACE_DIR

GOLD_YIELD_FILE = BASE_DIR / "data" / "gold" / "features" / "features_yield_prediction.parquet"
MODELS_DIR = BASE_DIR / "data" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")
print(f"Cargando dataset de rendimiento desde: {GOLD_YIELD_FILE}")


Cargando dataset de rendimiento desde: C:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\Agrostat_app\data\gold\features\features_yield_prediction.parquet


## 2. Carga y Exploración de Variables del Feature Store
Cargamos la matriz con variables fisicoquímicas, edafológicas, biométricas y climáticas de los lotes de cultivo.


In [2]:
# Carga del Parquet
df_yield = pd.read_parquet(GOLD_YIELD_FILE)
print(f"Dimensiones de la matriz de features: {df_yield.shape[0]} lotes, {df_yield.shape[1]} atributos")
display(df_yield[[
    "batch_id", "lote_id", "calibre_promedio", "grados_brix", "ph_suelo",
    "precipitacion_mm", "temperatura_celsius", "rendimiento_kg_ha", "tasa_exportabilidad", "clase_exportacion"
]].head())


Dimensiones de la matriz de features: 29 lotes, 21 atributos


,batch_id,lote_id,calibre_promedio,grados_brix,ph_suelo,precipitacion_mm,temperatura_celsius,rendimiento_kg_ha,tasa_exportabilidad,clase_exportacion
0,BATCH-2026-0001,LOTE-AGUACATE-02,40.7,10.0,6.96,38.9,19.5,12235.22,0.7892,ESTANDAR_EXPORT
1,BATCH-2026-0002,LOTE-CAFE-03,48.2,15.9,7.09,43.1,18.9,12027.48,0.8152,PREMIUM_EXPORT
2,BATCH-2026-0003,LOTE-AGUACATE-02,33.0,9.6,5.90,37.7,27.6,10477.93,0.8704,ESTANDAR_EXPORT
3,BATCH-2026-0004,LOTE-AGUACATE-02,45.3,14.9,5.66,29.3,20.4,8467.07,0.9198,PREMIUM_EXPORT
4,BATCH-2026-0005,LOTE-PALMA-01,34.2,11.1,6.67,18.2,19.1,9801.06,0.8255,PREMIUM_EXPORT


## 3. Preparación de Matrices $X$ e $y$ para Regresión Supervisada
Definimos los predictores y la variable objetivo: Rendimiento por Hectárea (`rendimiento_kg_ha`).


In [3]:
# Definición de Features
feature_cols = [
    "calibre_promedio", "grados_brix", "ph_suelo", "humedad_relativa",
    "precipitacion_mm", "temperatura_celsius", "ratio_brix_calibre",
    "indice_estres_hidrico", "indice_calidad_suelo"
]

X = df_yield[feature_cols].copy()
y = df_yield["rendimiento_kg_ha"].copy()

# División Train / Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Matriz de Entrenamiento: {X_train.shape[0]} muestras | Matriz de Evaluación: {X_test.shape[0]} muestras")


Matriz de Entrenamiento: 23 muestras | Matriz de Evaluación: 6 muestras


## 4. Benchmark de Modelos de Regresión mediante 5-Fold Cross Validation
Comparamos:
1. **Ridge Regression**: Modelo lineal regularizado $L_2$ como baseline.
2. **Random Forest Regressor**: Ensamble no lineal basado en árboles independientes (Bagging).
3. **Gradient Boosting Regressor**: Ensamble secuencial basado en optimización de gradiente (Boosting).


In [4]:
# Definir modelos a comparar
models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.08, random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
benchmark_results = []

for name, model in models.items():
    r2_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2")
    neg_rmse_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
    
    # Entrenar en todo el conjunto de entrenamiento
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)
    
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_mae = mean_absolute_error(y_test, y_pred_test)
    
    benchmark_results.append({
        "Algoritmo": name,
        "CV R² (Mean)": round(float(np.mean(r2_scores)), 3),
        "CV RMSE (Mean)": round(float(-np.mean(neg_rmse_scores)), 2),
        "Test R²": round(test_r2, 3),
        "Test RMSE": round(test_rmse, 2),
        "Test MAE": round(test_mae, 2)
    })

benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)


,Algoritmo,CV R² (Mean),CV RMSE (Mean),Test R²,Test RMSE,Test MAE
0,Ridge Regression,-1.849,2232.21,-0.597,2214.37,2015.87
1,Random Forest,-1.893,2193.77,-0.574,2198.63,1982.60
2,Gradient Boosting,-3.946,2714.70,-0.579,2202.27,1974.18


## 5. Análisis de Importancia de Variables (Feature Importance)
Examinamos cuáles atributos tienen mayor impacto en el rendimiento agrícola según el modelo Random Forest.


In [5]:
best_rf = models["Random Forest"]
importances = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=importances.values, y=importances.index, palette="mako")
plt.title("Importancia Relativa de Características en Rendimiento Agrícola (Random Forest MDI)", fontsize=12)
plt.xlabel("Importancia Relativa (Gini / Impureza)", fontsize=11)
plt.ylabel("Atributo", fontsize=11)
plt.tight_layout()
plt.show()


C:\Users\ADAN\AppData\Local\Temp\ipykernel_2056\3840318832.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=importances.values, y=importances.index, palette="mako")
C:\Users\ADAN\AppData\Local\Temp\ipykernel_2056\3840318832.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Clasificación Supervisada de Calidad de Exportación
Entrenamos un clasificador para categorizar el lote según estándares internacionales de exportación.


In [6]:
# Target de Clasificación
y_class = df_yield["clase_exportacion"]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_class, test_size=0.25, random_state=42, stratify=y_class)

clf = RandomForestClassifier(n_estimators=80, max_depth=4, random_state=42)
clf.fit(X_train_c, y_train_c)
y_pred_c = clf.predict(X_test_c)

print("--- Reporte de Clasificación de Calidad Exportable ---")
print(classification_report(y_test_c, y_pred_c))

# Matriz de Confusión Visual
labels = sorted(list(y_class.unique()))
cm = confusion_matrix(y_test_c, y_pred_c, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=labels, yticklabels=labels)
plt.title("Matriz de Confusión: Clasificación de Lotes de Exportación", fontsize=12)
plt.xlabel("Predicción", fontsize=11)
plt.ylabel("Clase Real", fontsize=11)
plt.tight_layout()
plt.show()


--- Reporte de Clasificación de Calidad Exportable ---
                 precision    recall  f1-score   support

ESTANDAR_EXPORT       0.67      0.67      0.67         3
 PREMIUM_EXPORT       0.80      0.80      0.80         5

       accuracy                           0.75         8
      macro avg       0.73      0.73      0.73         8
   weighted avg       0.75      0.75      0.75         8



C:\Users\ADAN\AppData\Local\Temp\ipykernel_2056\3656787608.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Serialización y Registro del Modelo en la Capa Gold
Guardamos el modelo entrenado y su archivo de metadatos en `data/models/`.


In [7]:
model_output_dir = MODELS_DIR / "yield_rf_production"
model_output_dir.mkdir(parents=True, exist_ok=True)
model_file = model_output_dir / "random_forest_yield_v1.pkl"
meta_file = model_output_dir / "metadata.json"

joblib.dump(best_rf, model_file)

metadata_payload = {
    "model_name": "RandomForestYieldRegressor",
    "version": "1.0.0",
    "algorithm": "RandomForestRegressor",
    "n_estimators": 100,
    "features": feature_cols,
    "target": "rendimiento_kg_ha",
    "metrics": benchmark_df[benchmark_df["Algoritmo"] == "Random Forest"].to_dict(orient="records")[0],
    "author": "AgroStats ML Engine"
}

with open(meta_file, "w", encoding="utf-8") as f:
    json.dump(metadata_payload, f, indent=2)

print(f"[OK] Modelo serializado en: {model_file}")
print(f"[OK] Metadatos guardados en: {meta_file}")


[OK] Modelo serializado en: C:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\Agrostat_app\data\models\yield_rf_production\random_forest_yield_v1.pkl
[OK] Metadatos guardados en: C:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\Agrostat_app\data\models\yield_rf_production\metadata.json
